# Tahap 01 — Data Inventory, Metadata, dan Source Validation

## Judul Project
**Analisis Komputasional Topik Pidato Presiden Prabowo pada Forum Nasional dan Internasional Menggunakan Manual Coding dan BERTopic**

## Tujuan Notebook
Notebook ini digunakan untuk:
1. Menyiapkan path project secara aman.
2. Memvalidasi keberadaan 6 file TXT pada `data/raw/`.
3. Membaca file TXT.
4. Mengekstraksi `source_url` jika tersedia.
5. Membuat `speech_id`.
6. Membuat metadata pidato.
7. Menghitung jumlah kata, karakter, dan baris.
8. Membuat `speech_raw_master.csv`.
9. Membuat `speech_metadata.csv`.
10. Membuat `data_inventory_summary.csv`.
11. Menyimpan output ke `data/interim/` dan `reports/tables/`.
12. Menambahkan validasi eksplisit agar tidak ada kolom yang diasumsikan tanpa dicek.

In [1]:
# ============================================================
# Import Library
# ============================================================

from __future__ import annotations

from pathlib import Path
from urllib.parse import urlparse
from datetime import datetime
from IPython.display import display

import hashlib
import re
import sys

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

print("Library berhasil di-import.")
print(f"Python version : {sys.version}")
print(f"Pandas version : {pd.__version__}")

Library berhasil di-import.
Python version : 3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 13:17:27) [MSC v.1929 64 bit (AMD64)]
Pandas version : 2.2.2


## 1. Setup Path Project

Struktur folder yang diharapkan:

```text
project-root/
├── data/
│   ├── raw/
│   └── interim/
├── notebooks/
├── reports/
│   └── tables/
└── README.md
```

Semua file TXT naskah pidato harus berada di:

```text
data/raw/
```

In [2]:
# ============================================================
# Setup Path Project
# ============================================================

def find_project_root(start_path: Path | None = None) -> Path:
    """
    Mendeteksi root folder project secara aman.

    Strategi:
    1. Mulai dari current working directory.
    2. Cek folder saat ini dan semua parent folder.
    3. Root project dipilih jika memiliki salah satu penanda:
       - folder data/raw
       - folder notebooks dan data
       - folder .git
    4. Jika tidak ditemukan, gunakan current working directory sebagai fallback.
    """
    if start_path is None:
        start_path = Path.cwd().resolve()
    else:
        start_path = Path(start_path).resolve()

    candidate_paths = [start_path] + list(start_path.parents)

    for candidate in candidate_paths:
        has_data_raw = (candidate / "data" / "raw").exists()
        has_project_dirs = (candidate / "notebooks").exists() and (candidate / "data").exists()
        has_git = (candidate / ".git").exists()

        if has_data_raw or has_project_dirs or has_git:
            return candidate

    return start_path


PROJECT_ROOT = find_project_root()

RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORT_TABLE_DIR = REPORTS_DIR / "tables"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

# Membuat folder output jika belum tersedia
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
REPORT_TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("Project path berhasil disiapkan.")
print(f"Current working directory : {Path.cwd().resolve()}")
print(f"PROJECT_ROOT              : {PROJECT_ROOT}")
print(f"RAW_DIR                   : {RAW_DIR}")
print(f"INTERIM_DIR               : {INTERIM_DIR}")
print(f"REPORT_TABLE_DIR          : {REPORT_TABLE_DIR}")

if not RAW_DIR.exists():
    raise FileNotFoundError(
        f"Folder data/raw belum ditemukan pada path berikut:\n{RAW_DIR}\n\n"
        "Solusi:\n"
        "1. Buat folder data/raw pada root project.\n"
        "2. Letakkan keenam file TXT naskah pidato ke folder data/raw.\n"
        "3. Jalankan ulang notebook dari awal."
    )

Project path berhasil disiapkan.
Current working directory : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\notebooks
PROJECT_ROOT              : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech
RAW_DIR                   : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\raw
INTERIM_DIR               : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\interim
REPORT_TABLE_DIR          : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\reports\tables


## 2. Definisi File Raw yang Diharapkan

Dataset raw terdiri dari 6 file TXT:

1. `NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt`
2. `NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt`
3. `NASKAH-PIDATO-PRABOWO-PERESMIAN-INFRASTRUKTUR-ENERGI.txt`
4. `NASKAH-PIDATO-PRABOWO-PERESMIAN-166-SEKOLAH.txt`
5. `NASKAH-PIDATO-PRABOWO-WORLD-ECONOMIC-FORUM.txt`
6. `NASKAH-PIDATO-PRABOWO-PBB-80.txt`

In [3]:
# ============================================================
# Daftar File Raw yang Diharapkan
# ============================================================

EXPECTED_RAW_FILES = [
    "NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt",
    "NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt",
    "NASKAH-PIDATO-PRABOWO-PERESMIAN-INFRASTRUKTUR-ENERGI.txt",
    "NASKAH-PIDATO-PRABOWO-PERESMIAN-166-SEKOLAH.txt",
    "NASKAH-PIDATO-PRABOWO-WORLD-ECONOMIC-FORUM.txt",
    "NASKAH-PIDATO-PRABOWO-PBB-80.txt",
]

if len(EXPECTED_RAW_FILES) != len(set(EXPECTED_RAW_FILES)):
    raise ValueError("Terdapat duplikasi nama file pada EXPECTED_RAW_FILES.")

print(f"Jumlah file raw yang diharapkan: {len(EXPECTED_RAW_FILES)}")
for file_name in EXPECTED_RAW_FILES:
    print(f"- {file_name}")

Jumlah file raw yang diharapkan: 6
- NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt
- NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt
- NASKAH-PIDATO-PRABOWO-PERESMIAN-INFRASTRUKTUR-ENERGI.txt
- NASKAH-PIDATO-PRABOWO-PERESMIAN-166-SEKOLAH.txt
- NASKAH-PIDATO-PRABOWO-WORLD-ECONOMIC-FORUM.txt
- NASKAH-PIDATO-PRABOWO-PBB-80.txt


## 3. Preflight Check

Cell ini memastikan variabel penting sudah tersedia sebelum proses validasi file dilakukan.

In [4]:
# ============================================================
# Preflight Check
# ============================================================

required_variables = [
    "PROJECT_ROOT",
    "RAW_DIR",
    "INTERIM_DIR",
    "REPORT_TABLE_DIR",
    "EXPECTED_RAW_FILES",
]

missing_variables = [
    variable_name for variable_name in required_variables
    if variable_name not in globals()
]

if missing_variables:
    raise NameError(
        "Variabel berikut belum tersedia: "
        + ", ".join(missing_variables)
        + "\nJalankan notebook dari awal atau jalankan kembali cell setup path."
    )

print("Preflight check berhasil.")
print(f"RAW_DIR tersedia: {RAW_DIR}")

Preflight check berhasil.
RAW_DIR tersedia: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\raw


## 4. Validasi Keberadaan File Raw

Validasi yang dilakukan:
1. Mengecek keberadaan file.
2. Mengecek apakah path tersebut benar-benar file.
3. Mengecek ukuran file.
4. Mengecek apakah file kosong.
5. Menampilkan file TXT aktual pada folder `data/raw/`.

In [6]:
# ============================================================
# Validasi Keberadaan File Raw
# ============================================================

def validate_raw_files(raw_dir: Path, expected_files: list[str]) -> pd.DataFrame:
    """
    Memvalidasi keberadaan file raw berdasarkan daftar file yang diharapkan.
    """
    records = []

    for file_name in expected_files:
        file_path = raw_dir / file_name

        exists = file_path.exists()
        is_file = file_path.is_file() if exists else False
        size_bytes = file_path.stat().st_size if is_file else 0

        records.append({
            "file_name": file_name,
            "file_path": str(file_path),
            "exists": exists,
            "is_file": is_file,
            "size_bytes": size_bytes,
            "is_empty": size_bytes == 0,
        })

    return pd.DataFrame(records)


actual_txt_files = sorted([path.name for path in RAW_DIR.glob("*.txt")])

print("File TXT yang ditemukan pada data/raw/:")
if actual_txt_files:
    for file_name in actual_txt_files:
        print(f"- {file_name}")
else:
    print("- Tidak ada file TXT ditemukan.")

raw_file_inventory_df = validate_raw_files(RAW_DIR, EXPECTED_RAW_FILES)

display(raw_file_inventory_df)

missing_files = raw_file_inventory_df.loc[
    raw_file_inventory_df["exists"] == False,
    "file_name"
].tolist()

not_file_items = raw_file_inventory_df.loc[
    (raw_file_inventory_df["exists"] == True) & (raw_file_inventory_df["is_file"] == False),
    "file_name"
].tolist()

empty_files = raw_file_inventory_df.loc[
    raw_file_inventory_df["is_empty"] == True,
    "file_name"
].tolist()

if missing_files:
    raise FileNotFoundError(
        "File berikut belum ditemukan pada folder data/raw/:\n"
        + "\n".join(f"- {file_name}" for file_name in missing_files)
        + "\n\nPastikan nama file sama persis, termasuk huruf besar/kecil dan tanda hubung."
    )

if not_file_items:
    raise ValueError(
        "Item berikut ditemukan tetapi bukan file:\n"
        + "\n".join(f"- {file_name}" for file_name in not_file_items)
    )

if empty_files:
    raise ValueError(
        "File berikut ditemukan tetapi kosong:\n"
        + "\n".join(f"- {file_name}" for file_name in empty_files)
    )

print("Validasi file raw berhasil: seluruh file ditemukan dan tidak kosong.")

File TXT yang ditemukan pada data/raw/:
- NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt
- NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt
- NASKAH-PIDATO-PRABOWO-PBB-80.txt
- NASKAH-PIDATO-PRABOWO-PERESMIAN-166-SEKOLAH.txt
- NASKAH-PIDATO-PRABOWO-PERESMIAN-INFRASTRUKTUR-ENERGI.txt
- NASKAH-PIDATO-PRABOWO-WORLD-ECONOMIC-FORUM.txt


,file_name,file_path,exists,is_file,size_bytes,is_empty
0,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\raw\NASKAH-PIDATO-PRABOWO-BRICS-LE...,True,True,1888,False
1,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\raw\NASKAH-PIDATO-PRABOWO-PANEN-RA...,True,True,24596,False
2,NASKAH-PIDATO-PRABOWO-PERESMIAN-INFRASTRUKTUR-ENERGI.txt,D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\raw\NASKAH-PIDATO-PRABOWO-PERESMIA...,True,True,18090,False
3,NASKAH-PIDATO-PRABOWO-PERESMIAN-166-SEKOLAH.txt,D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\raw\NASKAH-PIDATO-PRABOWO-PERESMIA...,True,True,24811,False
4,NASKAH-PIDATO-PRABOWO-WORLD-ECONOMIC-FORUM.txt,D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\raw\NASKAH-PIDATO-PRABOWO-WORLD-EC...,True,True,21485,False
5,NASKAH-PIDATO-PRABOWO-PBB-80.txt,D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\raw\NASKAH-PIDATO-PRABOWO-PBB-80.txt,True,True,11247,False


Validasi file raw berhasil: seluruh file ditemukan dan tidak kosong.


## 5. Fungsi Helper untuk Pembacaan dan Validasi Data

Fungsi helper digunakan agar proses pembacaan, ekstraksi URL, pembuatan ID, dan validasi kolom menjadi modular.

In [7]:
# ============================================================
# Helper Functions
# ============================================================

URL_PATTERN = re.compile(r"https?://[^\s]+", re.IGNORECASE)

INDONESIAN_MONTHS = {
    "januari": "01",
    "februari": "02",
    "maret": "03",
    "april": "04",
    "mei": "05",
    "juni": "06",
    "juli": "07",
    "agustus": "08",
    "september": "09",
    "oktober": "10",
    "november": "11",
    "desember": "12",
}

INDONESIAN_STOPWORDS_SAMPLE = {
    "yang", "dan", "di", "ke", "dari", "dengan", "untuk", "kita", "saya",
    "saudara", "para", "ini", "itu", "adalah", "dalam", "akan", "tidak",
    "pada", "sebagai", "karena", "bahwa", "juga", "semua", "rakyat"
}

ENGLISH_STOPWORDS_SAMPLE = {
    "the", "and", "of", "to", "in", "we", "are", "is", "that", "for",
    "with", "our", "this", "will", "as", "on", "be", "not", "have",
    "from", "by", "all", "must", "can"
}


def read_text_file(file_path: Path) -> tuple[str, str]:
    """
    Membaca file teks dengan beberapa fallback encoding.
    Mengembalikan tuple: (text, encoding_used).
    """
    encodings = ["utf-8", "utf-8-sig", "cp1252", "latin-1"]
    last_error = None

    for encoding in encodings:
        try:
            text = file_path.read_text(encoding=encoding)
            return text, encoding
        except UnicodeDecodeError as error:
            last_error = error

    raise UnicodeDecodeError(
        "unknown",
        b"",
        0,
        1,
        f"Gagal membaca file {file_path.name}. Error terakhir: {last_error}",
    )


def extract_urls(text: str) -> list[str]:
    """
    Mengekstraksi seluruh URL dari teks.
    """
    if not isinstance(text, str):
        return []

    urls = URL_PATTERN.findall(text)
    return [url.strip().rstrip(").,;]") for url in urls]


def extract_source_url(text: str) -> str | None:
    """
    Mengekstraksi source URL.
    Prioritas:
    1. URL pada baris yang mengandung 'Read more:'
    2. URL pertama yang ditemukan di teks
    """
    if not isinstance(text, str):
        return None

    for line in text.splitlines():
        if "read more:" in line.lower():
            urls = extract_urls(line)
            if urls:
                return urls[0]

    all_urls = extract_urls(text)
    if all_urls:
        return all_urls[0]

    return None


def remove_source_note(text: str) -> str:
    """
    Menghapus baris sumber seperti 'Read more: <url>'.
    """
    if not isinstance(text, str):
        return ""

    cleaned_lines = []

    for line in text.splitlines():
        if line.strip().lower().startswith("read more:"):
            continue
        cleaned_lines.append(line)

    return "\n".join(cleaned_lines).strip()


def normalize_whitespace(text: str) -> str:
    """
    Menormalkan whitespace berlebih tanpa mengubah substansi isi pidato.
    """
    if not isinstance(text, str):
        return ""

    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def count_words(text: str) -> int:
    """
    Menghitung jumlah kata menggunakan regex sederhana.
    """
    if not isinstance(text, str) or not text.strip():
        return 0

    tokens = re.findall(r"\b[\wÀ-ÿ’'-]+\b", text, flags=re.UNICODE)
    return len(tokens)


def count_lines(text: str) -> int:
    """
    Menghitung jumlah baris non-kosong.
    """
    if not isinstance(text, str) or not text.strip():
        return 0

    return len([line for line in text.splitlines() if line.strip()])


def create_content_hash(text: str) -> str:
    """
    Membuat SHA-256 hash untuk mendeteksi potensi duplikasi konten.
    """
    if not isinstance(text, str):
        text = ""

    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def create_speech_id(index: int, file_name: str) -> str:
    """
    Membuat speech_id deterministik berdasarkan urutan file dan nama file.
    """
    stem = Path(file_name).stem.upper()
    stem = stem.replace("NASKAH-PIDATO-PRABOWO-", "")
    stem = re.sub(r"[^A-Z0-9]+", "_", stem).strip("_")

    return f"SPCH_{index:03d}_{stem}"


def title_from_filename(file_name: str) -> str:
    """
    Membuat judul administratif dari nama file.
    """
    stem = Path(file_name).stem
    stem = stem.replace("NASKAH-PIDATO-PRABOWO-", "")
    return stem.replace("-", " ").title()


def infer_forum_scope_from_filename(file_name: str) -> tuple[str, str]:
    """
    Menginfer kategori forum dari nama file secara rule-based.
    """
    upper_name = file_name.upper()

    international_keywords = ["BRICS", "WORLD-ECONOMIC-FORUM", "PBB"]
    national_keywords = ["PANEN-RAYA", "PERESMIAN"]

    if any(keyword in upper_name for keyword in international_keywords):
        return "international", "filename_contains_international_forum_keyword"

    if any(keyword in upper_name for keyword in national_keywords):
        return "national", "filename_contains_national_event_keyword"

    return "unknown", "no_matching_keyword"


def estimate_language(text: str) -> str:
    """
    Estimasi bahasa sederhana berbasis stopword sample.
    Hasil bersifat indikatif, bukan language detection formal.
    """
    if not isinstance(text, str) or not text.strip():
        return "unknown"

    words = re.findall(r"\b[a-zA-ZÀ-ÿ']+\b", text.lower(), flags=re.UNICODE)

    if not words:
        return "unknown"

    id_count = sum(1 for word in words if word in INDONESIAN_STOPWORDS_SAMPLE)
    en_count = sum(1 for word in words if word in ENGLISH_STOPWORDS_SAMPLE)

    if id_count == 0 and en_count == 0:
        return "unknown"

    if id_count > en_count * 1.5:
        return "id"

    if en_count > id_count * 1.5:
        return "en"

    return "mixed"


def parse_domain(source_url: str | None) -> str | None:
    """
    Mengambil domain dari source_url.
    """
    if not source_url:
        return None

    parsed = urlparse(source_url)
    return parsed.netloc.lower() if parsed.netloc else None


def validate_source_url(source_url: str | None) -> str:
    """
    Validasi sumber secara offline.
    Tidak melakukan HTTP request.
    """
    if not source_url:
        return "MISSING_SOURCE_URL"

    parsed = urlparse(source_url)

    if parsed.scheme not in {"http", "https"} or not parsed.netloc:
        return "INVALID_URL_FORMAT"

    domain = parsed.netloc.lower()

    if domain == "setkab.go.id" or domain.endswith(".setkab.go.id"):
        return "VALID_SETKAB_DOMAIN_OFFLINE"

    return "URL_FOUND_NON_SETKAB_DOMAIN"


def extract_date_from_text_or_url(text: str, source_url: str | None) -> tuple[str | None, str]:
    """
    Mengekstraksi tanggal dari source_url atau teks jika tersedia.
    Format output tanggal: YYYY-MM-DD.
    """
    candidates = []

    if source_url:
        candidates.append(source_url.lower())

    if isinstance(text, str):
        candidates.append(text.lower())

    date_pattern = re.compile(
        r"(\d{1,2})[-\s]+("
        + "|".join(INDONESIAN_MONTHS.keys())
        + r")[-\s]+(\d{4})",
        re.IGNORECASE,
    )

    for candidate in candidates:
        match = date_pattern.search(candidate)

        if match:
            day, month_name, year = match.groups()
            month = INDONESIAN_MONTHS[month_name.lower()]
            iso_date = f"{year}-{month}-{int(day):02d}"
            return iso_date, "source_url_or_text_pattern"

    return None, "date_not_found"


def require_columns(df: pd.DataFrame, required_columns: list[str], df_name: str = "DataFrame") -> None:
    """
    Memastikan DataFrame memiliki kolom wajib sebelum diproses.
    """
    missing_columns = [column for column in required_columns if column not in df.columns]

    if missing_columns:
        raise ValueError(
            f"{df_name} tidak memiliki kolom wajib: {missing_columns}. "
            f"Kolom tersedia: {list(df.columns)}"
        )


def reorder_columns(df: pd.DataFrame, ordered_columns: list[str], df_name: str = "DataFrame") -> pd.DataFrame:
    """
    Mengurutkan kolom setelah memastikan seluruh kolom tersedia.
    """
    require_columns(df, ordered_columns, df_name)
    return df[ordered_columns].copy()


print("Helper functions berhasil dibuat.")

Helper functions berhasil dibuat.


## 6. Membaca File TXT dan Membuat Dataset Raw Master

Dataset master menyimpan teks asli (`text_raw`), teks isi pidato tanpa baris sumber (`text_body`), metadata administratif, statistik teks, dan hash konten.

In [8]:
# ============================================================
# Membaca File TXT dan Membuat Raw Master
# ============================================================

raw_records = []

for index, file_name in enumerate(EXPECTED_RAW_FILES, start=1):
    file_path = RAW_DIR / file_name

    text_raw, encoding_used = read_text_file(file_path)
    text_raw = normalize_whitespace(text_raw)

    source_url = extract_source_url(text_raw)
    source_domain = parse_domain(source_url)
    source_validation_status = validate_source_url(source_url)

    text_body = remove_source_note(text_raw)
    text_body = normalize_whitespace(text_body)

    event_date, event_date_source = extract_date_from_text_or_url(text_body, source_url)
    forum_scope, forum_scope_rule = infer_forum_scope_from_filename(file_name)

    raw_records.append({
        "speech_id": create_speech_id(index, file_name),
        "file_name": file_name,
        "file_path": str(file_path),
        "file_size_bytes": file_path.stat().st_size,
        "encoding_used": encoding_used,
        "speech_title_from_filename": title_from_filename(file_name),
        "forum_scope_inferred": forum_scope,
        "forum_scope_rule": forum_scope_rule,
        "event_date": event_date,
        "event_date_source": event_date_source,
        "source_url": source_url,
        "source_domain": source_domain,
        "source_validation_status": source_validation_status,
        "url_count": len(extract_urls(text_raw)),
        "language_estimate": estimate_language(text_body),
        "text_raw": text_raw,
        "text_body": text_body,
        "char_count_raw": len(text_raw),
        "char_count_body": len(text_body),
        "word_count_body": count_words(text_body),
        "line_count_body": count_lines(text_body),
        "content_sha256": create_content_hash(text_body),
        "processed_at": datetime.now().isoformat(timespec="seconds"),
    })

speech_raw_master_df = pd.DataFrame(raw_records)

RAW_MASTER_COLUMNS = [
    "speech_id",
    "file_name",
    "file_path",
    "file_size_bytes",
    "encoding_used",
    "speech_title_from_filename",
    "forum_scope_inferred",
    "forum_scope_rule",
    "event_date",
    "event_date_source",
    "source_url",
    "source_domain",
    "source_validation_status",
    "url_count",
    "language_estimate",
    "text_raw",
    "text_body",
    "char_count_raw",
    "char_count_body",
    "word_count_body",
    "line_count_body",
    "content_sha256",
    "processed_at",
]

speech_raw_master_df = reorder_columns(
    speech_raw_master_df,
    RAW_MASTER_COLUMNS,
    df_name="speech_raw_master_df",
)

display(speech_raw_master_df.head())
print(f"Jumlah dokumen pidato yang berhasil dibaca: {len(speech_raw_master_df)}")

,speech_id,file_name,file_path,file_size_bytes,encoding_used,speech_title_from_filename,forum_scope_inferred,forum_scope_rule,event_date,event_date_source,source_url,source_domain,source_validation_status,url_count,language_estimate,text_raw,text_body,char_count_raw,char_count_body,word_count_body,line_count_body,content_sha256,processed_at
0,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\raw\NASKAH-PIDATO-PRABOWO-BRICS-LE...,1888,utf-8,Brics Leaders,international,filename_contains_international_forum_keyword,2025-09-08,source_url_or_text_pattern,https://setkab.go.id/sambutan-presiden-republik-indonesia-pada-brics-leaders-virtual-meeting-melalui-video-conferenc...,setkab.go.id,VALID_SETKAB_DOMAIN_OFFLINE,1,en,Distinguished Leaders of BRICS.\n\nIt is indeed a great honor for me to join this very important meeting.\n\nIndones...,Distinguished Leaders of BRICS.\n\nIt is indeed a great honor for me to join this very important meeting.\n\nIndones...,1866,1642,256,8,a2f66ccd4a1d6ba1b4793089989eb09607add274f280505c062aa477bdaface9,2026-06-11T21:24:37
1,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\raw\NASKAH-PIDATO-PRABOWO-PANEN-RA...,24596,utf-8,Panen Raya,national,filename_contains_national_event_keyword,2026-01-07,source_url_or_text_pattern,https://setkab.go.id/sambutan-presiden-republik-indonesia-pada-panen-raya-dan-pengumuman-swasembada-pangan-di-desa-k...,setkab.go.id,VALID_SETKAB_DOMAIN_OFFLINE,1,id,"Bismillahirrahmanirrahim.\n\nAssalamu’alaikum warahmatullahi wabarakatuh,\nSelamat siang,\nSalam sejahtera bagi kita...","Bismillahirrahmanirrahim.\n\nAssalamu’alaikum warahmatullahi wabarakatuh,\nSelamat siang,\nSalam sejahtera bagi kita...",24418,24220,3417,82,2fe161589989ad5494b22841bb0c3bc26977e64fecbd0c9994e413e47e42db77,2026-06-11T21:24:37
2,SPCH_003_PERESMIAN_INFRASTRUKTUR_ENERGI,NASKAH-PIDATO-PRABOWO-PERESMIAN-INFRASTRUKTUR-ENERGI.txt,D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\raw\NASKAH-PIDATO-PRABOWO-PERESMIA...,18090,utf-8,Peresmian Infrastruktur Energi,national,filename_contains_national_event_keyword,None,date_not_found,None,None,MISSING_SOURCE_URL,0,id,"Bismillahirrahmanirrahim.\n\nAssalamu’alaikum warahmatullahi wabarakatuh,\nSelamat sore,\nSalam sejahtera bagi kita ...","Bismillahirrahmanirrahim.\n\nAssalamu’alaikum warahmatullahi wabarakatuh,\nSelamat sore,\nSalam sejahtera bagi kita ...",17990,17990,2543,54,881ebf65c5148e942e7fb142ee7950d6d91a5381cdbb4683c7b96a3beb024fda,2026-06-11T21:24:37
3,SPCH_004_PERESMIAN_166_SEKOLAH,NASKAH-PIDATO-PRABOWO-PERESMIAN-166-SEKOLAH.txt,D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\raw\NASKAH-PIDATO-PRABOWO-PERESMIA...,24811,utf-8,Peresmian 166 Sekolah,national,filename_contains_national_event_keyword,None,date_not_found,https://setkab.go.id/sambutan-presiden-republik-indonesia-pada-peresmian-166-sekolah-rakyat-di-34-provinsi-di-balai-...,setkab.go.id,VALID_SETKAB_DOMAIN_OFFLINE,1,id,"Bismillahirrahmanirrahim,\n\nAssalamu’alaikum warahmatullahi wabarakatuh,\nSelamat pagi oh sudah selamat siang,\nSal...","Bismillahirrahmanirrahim,\n\nAssalamu’alaikum warahmatullahi wabarakatuh,\nSelamat pagi oh sudah selamat siang,\nSal...",24690,24459,3484,60,1452ea654813e9675ec6c863ef5f68f47d796cdbf5a01f7171d75abe5b0341cf,2026-06-11T21:24:37
4,SPCH_005_WORLD_ECONOMIC_FORUM,NASKAH-PIDATO-PRABOWO-WORLD-ECONOMIC-FORUM.txt,D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\raw\NASKAH-PIDATO-PRABOWO-WORLD-EC...,21485,utf-8,World Economic Forum,international,filename_contains_international_forum_keyword,2026-01-22,source_url_or_text_pattern,https://setkab.go.id/sambutan-presiden-republik-indonesia-pada-world-economic-forum-wef-annual-meeting-2026-di-londo...,setkab.go.id,VALID_SETKAB_DOMAIN_OFFLINE,1,en,"Disti

Jumlah dokumen pidato yang berhasil dibaca: 6


## 7. Validasi `speech_id`, Jumlah Kata, Jumlah Karakter, dan Duplikasi Konten

In [9]:
# ============================================================
# Validasi Struktur Dasar Dataset
# ============================================================

required_validation_columns = [
    "speech_id",
    "file_name",
    "word_count_body",
    "char_count_body",
    "content_sha256",
    "source_validation_status",
]

require_columns(
    speech_raw_master_df,
    required_validation_columns,
    df_name="speech_raw_master_df",
)

if speech_raw_master_df["speech_id"].isna().any():
    raise ValueError("Terdapat speech_id yang kosong.")

if speech_raw_master_df["speech_id"].duplicated().any():
    duplicated_ids = speech_raw_master_df.loc[
        speech_raw_master_df["speech_id"].duplicated(keep=False),
        "speech_id",
    ].tolist()
    raise ValueError(f"Terdapat speech_id duplikat: {duplicated_ids}")

empty_text_df = speech_raw_master_df[
    (speech_raw_master_df["word_count_body"] <= 0) |
    (speech_raw_master_df["char_count_body"] <= 0)
]

if not empty_text_df.empty:
    display(empty_text_df[["speech_id", "file_name", "word_count_body", "char_count_body"]])
    raise ValueError("Terdapat dokumen dengan text_body kosong atau tidak valid.")

duplicate_content_df = speech_raw_master_df[
    speech_raw_master_df["content_sha256"].duplicated(keep=False)
]

if not duplicate_content_df.empty:
    print("Peringatan: terdapat potensi duplikasi konten berdasarkan hash.")
    display(duplicate_content_df[["speech_id", "file_name", "content_sha256"]])
else:
    print("Tidak ditemukan duplikasi konten berdasarkan SHA-256 hash.")

source_status_summary = (
    speech_raw_master_df["source_validation_status"]
    .value_counts(dropna=False)
    .rename_axis("source_validation_status")
    .reset_index(name="document_count")
)

display(source_status_summary)

print("Validasi struktur dasar dataset berhasil.")

Tidak ditemukan duplikasi konten berdasarkan SHA-256 hash.


,source_validation_status,document_count
0,VALID_SETKAB_DOMAIN_OFFLINE,4
1,MISSING_SOURCE_URL,2


Validasi struktur dasar dataset berhasil.


## 8. Membuat Metadata Pidato

Metadata dibuat dengan prinsip kehati-hatian:
- Kategori forum merupakan inferensi dari nama file.
- Estimasi bahasa bersifat indikatif.
- URL dan tanggal hanya diisi jika ditemukan.
- Dokumen yang belum lengkap diberi `quality_flags`.

In [10]:
# ============================================================
# Membuat Speech Metadata
# ============================================================

METADATA_COLUMNS = [
    "speech_id",
    "file_name",
    "speech_title_from_filename",
    "forum_scope_inferred",
    "forum_scope_rule",
    "event_date",
    "event_date_source",
    "language_estimate",
    "source_url",
    "source_domain",
    "source_validation_status",
    "url_count",
    "file_size_bytes",
    "char_count_body",
    "word_count_body",
    "line_count_body",
    "content_sha256",
    "processed_at",
]

require_columns(
    speech_raw_master_df,
    METADATA_COLUMNS,
    df_name="speech_raw_master_df",
)

speech_metadata_df = speech_raw_master_df[METADATA_COLUMNS].copy()

speech_metadata_df["has_source_url"] = speech_metadata_df["source_url"].notna()
speech_metadata_df["has_event_date"] = speech_metadata_df["event_date"].notna()
speech_metadata_df["is_text_length_valid"] = (
    (speech_metadata_df["word_count_body"] > 0) &
    (speech_metadata_df["char_count_body"] > 0)
)


def build_quality_flags(row: pd.Series) -> str:
    """
    Membuat quality flag per dokumen berdasarkan kondisi eksplisit.
    """
    flags = []

    if not row["has_source_url"]:
        flags.append("MISSING_SOURCE_URL")

    if not row["has_event_date"]:
        flags.append("MISSING_EVENT_DATE")

    if row["word_count_body"] < 100:
        flags.append("LOW_WORD_COUNT_LT_100")

    if row["source_validation_status"] == "INVALID_URL_FORMAT":
        flags.append("INVALID_URL_FORMAT")

    if not row["is_text_length_valid"]:
        flags.append("INVALID_TEXT_LENGTH")

    if not flags:
        return "OK"

    return ";".join(flags)


speech_metadata_df["quality_flags"] = speech_metadata_df.apply(build_quality_flags, axis=1)

FINAL_METADATA_COLUMNS = [
    "speech_id",
    "file_name",
    "speech_title_from_filename",
    "forum_scope_inferred",
    "forum_scope_rule",
    "event_date",
    "event_date_source",
    "language_estimate",
    "source_url",
    "source_domain",
    "source_validation_status",
    "url_count",
    "has_source_url",
    "has_event_date",
    "is_text_length_valid",
    "file_size_bytes",
    "char_count_body",
    "word_count_body",
    "line_count_body",
    "content_sha256",
    "quality_flags",
    "processed_at",
]

speech_metadata_df = reorder_columns(
    speech_metadata_df,
    FINAL_METADATA_COLUMNS,
    df_name="speech_metadata_df",
)

display(speech_metadata_df)
print(f"Jumlah baris metadata: {len(speech_metadata_df)}")

,speech_id,file_name,speech_title_from_filename,forum_scope_inferred,forum_scope_rule,event_date,event_date_source,language_estimate,source_url,source_domain,source_validation_status,url_count,has_source_url,has_event_date,is_text_length_valid,file_size_bytes,char_count_body,word_count_body,line_count_body,content_sha256,quality_flags,processed_at
0,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,Brics Leaders,international,filename_contains_international_forum_keyword,2025-09-08,source_url_or_text_pattern,en,https://setkab.go.id/sambutan-presiden-republik-indonesia-pada-brics-leaders-virtual-meeting-melalui-video-conferenc...,setkab.go.id,VALID_SETKAB_DOMAIN_OFFLINE,1,True,True,True,1888,1642,256,8,a2f66ccd4a1d6ba1b4793089989eb09607add274f280505c062aa477bdaface9,OK,2026-06-11T21:24:37
1,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,Panen Raya,national,filename_contains_national_event_keyword,2026-01-07,source_url_or_text_pattern,id,https://setkab.go.id/sambutan-presiden-republik-indonesia-pada-panen-raya-dan-pengumuman-swasembada-pangan-di-desa-k...,setkab.go.id,VALID_SETKAB_DOMAIN_OFFLINE,1,True,True,True,24596,24220,3417,82,2fe161589989ad5494b22841bb0c3bc26977e64fecbd0c9994e413e47e42db77,OK,2026-06-11T21:24:37
2,SPCH_003_PERESMIAN_INFRASTRUKTUR_ENERGI,NASKAH-PIDATO-PRABOWO-PERESMIAN-INFRASTRUKTUR-ENERGI.txt,Peresmian Infrastruktur Energi,national,filename_contains_national_event_keyword,None,date_not_found,id,None,None,MISSING_SOURCE_URL,0,False,False,True,18090,17990,2543,54,881ebf65c5148e942e7fb142ee7950d6d91a5381cdbb4683c7b96a3beb024fda,MISSING_SOURCE_URL;MISSING_EVENT_DATE,2026-06-11T21:24:37
3,SPCH_004_PERESMIAN_166_SEKOLAH,NASKAH-PIDATO-PRABOWO-PERESMIAN-166-SEKOLAH.txt,Peresmian 166 Sekolah,national,filename_contains_national_event_keyword,None,date_not_found,id,https://setkab.go.id/sambutan-presiden-republik-indonesia-pada-peresmian-166-sekolah-rakyat-di-34-provinsi-di-balai-...,setkab.go.id,VALID_SETKAB_DOMAIN_OFFLINE,1,True,False,True,24811,24459,3484,60,1452ea654813e9675ec6c863ef5f68f47d796cdbf5a01f7171d75abe5b0341cf,MISSING_EVENT_DATE,2026-06-11T21:24:37
4,SPCH_005_WORLD_ECONOMIC_FORUM,NASKAH-PIDATO-PRABOWO-WORLD-ECONOMIC-FORUM.txt,World Economic Forum,international,filename_contains_international_forum_keyword,2026-01-22,source_url_or_text_pattern,en,https://setkab.go.id/sambutan-presiden-republik-indonesia-pada-world-economic-forum-wef-annual-meeting-2026-di-londo...,setkab.go.id,VALID_SETKAB_DOMAIN_OFFLINE,1,True,True,True,21485,21135,3585,51,5c443ea53ec06c51d42ec156fbe3d3b5aa39f0917b65e264f46f20456f509d67,OK,2026-06-11T21:24:37
5,SPCH_006_PBB_80,NASKAH-PIDATO-PRABOWO-PBB-80.txt,Pbb 80,international,filename_contains_international_forum_keyword,None,date_not_found,en,None,None,MISSING_SOURCE_URL,0,False,False,True,11247,11116,1824,45,566692ee86957f24cc0e51f0be8236aa4f23788a93234549f932d49e21765a0b,MISSING_SOURCE_URL;MISSING_EVENT_DATE,2026-06-11T21:24:37


Jumlah baris metadata: 6


## 9. Membuat Data Inventory Summary

Tabel ini menggabungkan validasi file fisik, metadata pidato, statistik teks, dan flag kualitas data.

In [11]:
# ============================================================
# Membuat Data Inventory Summary
# ============================================================

inventory_required_columns = [
    "file_name",
    "file_path",
    "exists",
    "is_file",
    "size_bytes",
    "is_empty",
]

require_columns(
    raw_file_inventory_df,
    inventory_required_columns,
    df_name="raw_file_inventory_df",
)

metadata_summary_columns = [
    "speech_id",
    "file_name",
    "speech_title_from_filename",
    "forum_scope_inferred",
    "event_date",
    "language_estimate",
    "source_validation_status",
    "has_source_url",
    "has_event_date",
    "word_count_body",
    "char_count_body",
    "line_count_body",
    "quality_flags",
]

require_columns(
    speech_metadata_df,
    metadata_summary_columns,
    df_name="speech_metadata_df",
)

data_inventory_summary_df = raw_file_inventory_df.merge(
    speech_metadata_df[metadata_summary_columns],
    on="file_name",
    how="left",
    validate="one_to_one",
)

SUMMARY_COLUMNS = [
    "speech_id",
    "file_name",
    "speech_title_from_filename",
    "forum_scope_inferred",
    "event_date",
    "language_estimate",
    "exists",
    "is_file",
    "is_empty",
    "size_bytes",
    "word_count_body",
    "char_count_body",
    "line_count_body",
    "source_validation_status",
    "has_source_url",
    "has_event_date",
    "quality_flags",
    "file_path",
]

data_inventory_summary_df = reorder_columns(
    data_inventory_summary_df,
    SUMMARY_COLUMNS,
    df_name="data_inventory_summary_df",
)

display(data_inventory_summary_df)
print(f"Jumlah baris data inventory summary: {len(data_inventory_summary_df)}")

,speech_id,file_name,speech_title_from_filename,forum_scope_inferred,event_date,language_estimate,exists,is_file,is_empty,size_bytes,word_count_body,char_count_body,line_count_body,source_validation_status,has_source_url,has_event_date,quality_flags,file_path
0,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,Brics Leaders,international,2025-09-08,en,True,True,False,1888,256,1642,8,VALID_SETKAB_DOMAIN_OFFLINE,True,True,OK,D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\raw\NASKAH-PIDATO-PRABOWO-BRICS-LE...
1,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,Panen Raya,national,2026-01-07,id,True,True,False,24596,3417,24220,82,VALID_SETKAB_DOMAIN_OFFLINE,True,True,OK,D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\raw\NASKAH-PIDATO-PRABOWO-PANEN-RA...
2,SPCH_003_PERESMIAN_INFRASTRUKTUR_ENERGI,NASKAH-PIDATO-PRABOWO-PERESMIAN-INFRASTRUKTUR-ENERGI.txt,Peresmian Infrastruktur Energi,national,None,id,True,True,False,18090,2543,17990,54,MISSING_SOURCE_URL,False,False,MISSING_SOURCE_URL;MISSING_EVENT_DATE,D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\raw\NASKAH-PIDATO-PRABOWO-PERESMIA...
3,SPCH_004_PERESMIAN_166_SEKOLAH,NASKAH-PIDATO-PRABOWO-PERESMIAN-166-SEKOLAH.txt,Peresmian 166 Sekolah,national,None,id,True,True,False,24811,3484,24459,60,VALID_SETKAB_DOMAIN_OFFLINE,True,False,MISSING_EVENT_DATE,D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\raw\NASKAH-PIDATO-PRABOWO-PERESMIA...
4,SPCH_005_WORLD_ECONOMIC_FORUM,NASKAH-PIDATO-PRABOWO-WORLD-ECONOMIC-FORUM.txt,World Economic Forum,international,2026-01-22,en,True,True,False,21485,3585,21135,51,VALID_SETKAB_DOMAIN_OFFLINE,True,True,OK,D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\raw\NASKAH-PIDATO-PRABOWO-WORLD-EC...
5,SPCH_006_PBB_80,NASKAH-PIDATO-PRABOWO-PBB-80.txt,Pbb 80,international,None,en,True,True,False,11247,1824,11116,45,MISSING_SOURCE_URL,False,False,MISSING_SOURCE_URL;MISSING_EVENT_DATE,D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\raw\NASKAH-PIDATO-PRABOWO-PBB-80.txt


Jumlah baris data inventory summary: 6


## 10. Validasi Akhir Sebelum Penyimpanan Output

In [12]:
# ============================================================
# Validasi Akhir Sebelum Save
# ============================================================

expected_document_count = len(EXPECTED_RAW_FILES)

if len(speech_raw_master_df) != expected_document_count:
    raise ValueError(
        f"Jumlah speech_raw_master_df tidak sesuai. "
        f"Expected: {expected_document_count}, Actual: {len(speech_raw_master_df)}"
    )

if len(speech_metadata_df) != expected_document_count:
    raise ValueError(
        f"Jumlah speech_metadata_df tidak sesuai. "
        f"Expected: {expected_document_count}, Actual: {len(speech_metadata_df)}"
    )

if len(data_inventory_summary_df) != expected_document_count:
    raise ValueError(
        f"Jumlah data_inventory_summary_df tidak sesuai. "
        f"Expected: {expected_document_count}, Actual: {len(data_inventory_summary_df)}"
    )

require_columns(speech_raw_master_df, RAW_MASTER_COLUMNS, "speech_raw_master_df")
require_columns(speech_metadata_df, FINAL_METADATA_COLUMNS, "speech_metadata_df")
require_columns(data_inventory_summary_df, SUMMARY_COLUMNS, "data_inventory_summary_df")

for df_name, df in {
    "speech_raw_master_df": speech_raw_master_df,
    "speech_metadata_df": speech_metadata_df,
    "data_inventory_summary_df": data_inventory_summary_df,
}.items():
    if df["speech_id"].isna().any():
        raise ValueError(f"{df_name} memiliki speech_id kosong.")

    if df["file_name"].isna().any():
        raise ValueError(f"{df_name} memiliki file_name kosong.")

if (speech_metadata_df["word_count_body"] <= 0).any():
    raise ValueError("Terdapat word_count_body yang tidak valid.")

if (speech_metadata_df["char_count_body"] <= 0).any():
    raise ValueError("Terdapat char_count_body yang tidak valid.")

print("Validasi akhir berhasil. Dataset siap disimpan.")

Validasi akhir berhasil. Dataset siap disimpan.


## 11. Menyimpan Output ke `data/interim/` dan `reports/tables/`

Output:
1. `data/interim/speech_raw_master.csv`
2. `data/interim/speech_metadata.csv`
3. `reports/tables/data_inventory_summary.csv`

In [13]:
# ============================================================
# Save Output CSV
# ============================================================

speech_raw_master_path = INTERIM_DIR / "speech_raw_master.csv"
speech_metadata_path = INTERIM_DIR / "speech_metadata.csv"
data_inventory_summary_path = REPORT_TABLE_DIR / "data_inventory_summary.csv"

speech_raw_master_df.to_csv(speech_raw_master_path, index=False, encoding="utf-8-sig")
speech_metadata_df.to_csv(speech_metadata_path, index=False, encoding="utf-8-sig")
data_inventory_summary_df.to_csv(data_inventory_summary_path, index=False, encoding="utf-8-sig")

print("Output berhasil disimpan:")
print(f"1. {speech_raw_master_path}")
print(f"2. {speech_metadata_path}")
print(f"3. {data_inventory_summary_path}")

Output berhasil disimpan:
1. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\interim\speech_raw_master.csv
2. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\interim\speech_metadata.csv
3. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\reports\tables\data_inventory_summary.csv


## 12. Preview Output Akhir

In [14]:
# ============================================================
# Preview Output Akhir
# ============================================================

preview_columns = [
    "speech_id",
    "file_name",
    "forum_scope_inferred",
    "event_date",
    "language_estimate",
    "word_count_body",
    "char_count_body",
    "source_validation_status",
    "quality_flags",
]

require_columns(
    speech_metadata_df,
    preview_columns,
    df_name="speech_metadata_df",
)

display(speech_metadata_df[preview_columns])

print("Ringkasan jumlah dokumen berdasarkan forum_scope_inferred:")
forum_scope_summary_df = (
    speech_metadata_df["forum_scope_inferred"]
    .value_counts(dropna=False)
    .rename_axis("forum_scope_inferred")
    .reset_index(name="document_count")
)
display(forum_scope_summary_df)

print("Ringkasan quality_flags:")
quality_flags_summary_df = (
    speech_metadata_df["quality_flags"]
    .value_counts(dropna=False)
    .rename_axis("quality_flags")
    .reset_index(name="document_count")
)
display(quality_flags_summary_df)

print("Ringkasan source_validation_status:")
source_validation_summary_df = (
    speech_metadata_df["source_validation_status"]
    .value_counts(dropna=False)
    .rename_axis("source_validation_status")
    .reset_index(name="document_count")
)
display(source_validation_summary_df)

,speech_id,file_name,forum_scope_inferred,event_date,language_estimate,word_count_body,char_count_body,source_validation_status,quality_flags
0,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,international,2025-09-08,en,256,1642,VALID_SETKAB_DOMAIN_OFFLINE,OK
1,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,national,2026-01-07,id,3417,24220,VALID_SETKAB_DOMAIN_OFFLINE,OK
2,SPCH_003_PERESMIAN_INFRASTRUKTUR_ENERGI,NASKAH-PIDATO-PRABOWO-PERESMIAN-INFRASTRUKTUR-ENERGI.txt,national,None,id,2543,17990,MISSING_SOURCE_URL,MISSING_SOURCE_URL;MISSING_EVENT_DATE
3,SPCH_004_PERESMIAN_166_SEKOLAH,NASKAH-PIDATO-PRABOWO-PERESMIAN-166-SEKOLAH.txt,national,None,id,3484,24459,VALID_SETKAB_DOMAIN_OFFLINE,MISSING_EVENT_DATE
4,SPCH_005_WORLD_ECONOMIC_FORUM,NASKAH-PIDATO-PRABOWO-WORLD-ECONOMIC-FORUM.txt,international,2026-01-22,en,3585,21135,VALID_SETKAB_DOMAIN_OFFLINE,OK
5,SPCH_006_PBB_80,NASKAH-PIDATO-PRABOWO-PBB-80.txt,international,None,en,1824,11116,MISSING_SOURCE_URL,MISSING_SOURCE_URL;MISSING_EVENT_DATE


Ringkasan jumlah dokumen berdasarkan forum_scope_inferred:


,forum_scope_inferred,document_count
0,international,3
1,national,3


Ringkasan quality_flags:


,quality_flags,document_count
0,OK,3
1,MISSING_SOURCE_URL;MISSING_EVENT_DATE,2
2,MISSING_EVENT_DATE,1


Ringkasan source_validation_status:


,source_validation_status,document_count
0,VALID_SETKAB_DOMAIN_OFFLINE,4
1,MISSING_SOURCE_URL,2


## 13. Optional: Online URL Check

Secara default validasi dilakukan secara offline.  
Jika ingin mengecek HTTP status URL sumber, ubah:

```python
ENABLE_ONLINE_URL_CHECK = True
```

In [15]:
# ============================================================
# Optional Online URL Check
# ============================================================

from urllib.request import Request, urlopen
from urllib.error import URLError, HTTPError

ENABLE_ONLINE_URL_CHECK = False


def check_url_online(source_url: str | None, timeout: int = 10) -> dict:
    """
    Mengecek status online URL menggunakan urllib.
    Fungsi ini hanya berjalan jika ENABLE_ONLINE_URL_CHECK = True.
    """
    if not source_url:
        return {
            "online_check_status": "SKIPPED_NO_URL",
            "http_status_code": None,
            "online_error": None,
        }

    try:
        request = Request(
            source_url,
            headers={"User-Agent": "Mozilla/5.0"},
        )

        with urlopen(request, timeout=timeout) as response:
            return {
                "online_check_status": "ACCESSIBLE",
                "http_status_code": response.status,
                "online_error": None,
            }

    except HTTPError as error:
        return {
            "online_check_status": "HTTP_ERROR",
            "http_status_code": error.code,
            "online_error": str(error),
        }

    except URLError as error:
        return {
            "online_check_status": "URL_ERROR",
            "http_status_code": None,
            "online_error": str(error),
        }

    except Exception as error:
        return {
            "online_check_status": "UNKNOWN_ERROR",
            "http_status_code": None,
            "online_error": str(error),
        }


if ENABLE_ONLINE_URL_CHECK:
    online_check_records = []

    for _, row in speech_metadata_df.iterrows():
        result = check_url_online(row["source_url"])
        result["speech_id"] = row["speech_id"]
        result["file_name"] = row["file_name"]
        result["source_url"] = row["source_url"]
        online_check_records.append(result)

    online_source_check_df = pd.DataFrame(online_check_records)
    display(online_source_check_df)

    online_source_check_path = REPORT_TABLE_DIR / "online_source_check.csv"
    online_source_check_df.to_csv(online_source_check_path, index=False, encoding="utf-8-sig")

    print(f"Hasil online source check disimpan ke: {online_source_check_path}")
else:
    print("Online URL check dilewati. Ubah ENABLE_ONLINE_URL_CHECK = True jika ingin menjalankannya.")

Online URL check dilewati. Ubah ENABLE_ONLINE_URL_CHECK = True jika ingin menjalankannya.


## 14. Kesimpulan Tahap 01

Tahap 01 menghasilkan:
1. `speech_raw_master.csv`
2. `speech_metadata.csv`
3. `data_inventory_summary.csv`